In [ ]:
from os.path import join as opj
import json
import itertools
import tqdm
import pandas as pd
from typing import List
import numpy as np
import os
import copy
import torch

In [ ]:
from torchvision.transforms.functional import adjust_contrast, adjust_brightness
def normalize_im(x):
    return (adjust_brightness(adjust_contrast(x, 2), 2) * 255
                          ).to(torch.uint8)
def rearrange_viz(x):
    return einops.rearrange(x, "c h w -> h w c")

from ts2.data.transforms import HistologyTransform

In [ ]:
import einops
import yaml
from collections import namedtuple
import matplotlib.pyplot as plt
from PIL import Image
from collections import Counter

In [ ]:
hist_xform_params = yaml.safe_load("""
which_set: srh
base_aug_params:
    laser_noise_config: null
    get_third_channel_params:
        mode: three_channels
        subtracted_base: 0.07629394531
strong_aug_params:
    aug_list: []
    aug_prob: 0
""")
    
aug = HistologyTransform(**hist_xform_params)

In [ ]:
scsrh_repl_root = "/nfs/turbo/umms-tocho-snr/exp/chengjia/scsrh_repl_root2/"

train_slides = pd.read_csv(
    "/nfs/turbo/umms-tocho/data/data_splits/srh7v1/srh7v1_train.csv",        
    dtype=str)  

test_slides = pd.read_csv( 
    "/nfs/turbo/umms-tocho/data/data_splits/srh7v1/srh7v1_test.csv",        
    dtype=str)  

In [ ]:
def get_count(slide):
    slide_id = "_".join([slide["patient"],slide["mosaic"]])

    meta_fname = opj(scsrh_repl_root,  f"{slide_id}_meta.json")

    with open(meta_fname) as fd:
        meta_s = json.load(fd)

    celltypes = [ms["celltype"] for ms in meta_s["cells"]]

    count = dict(Counter(celltypes))
    count.update({"label": slide["label"]})

    return count


In [ ]:
train_ct = [get_count(r) for i, r in train_slides.iterrows()]
test_ct = [get_count(r) for i, r in test_slides.iterrows()]

In [ ]:
print(pd.DataFrame(train_ct).groupby("label").sum())
print(pd.DataFrame(test_ct).groupby("label").sum())

In [ ]:
sampled_train_slides = train_slides.groupby("label").sample(5).reset_index()

In [ ]:
def get_mean(slide):
    slide_id = "_".join([slide["patient"],slide["mosaic"]])

    meta_fname = opj(scsrh_repl_root,  f"{slide_id}_meta.json")

    with open(meta_fname) as fd:
        meta_s = json.load(fd)

    celltypes = [ms["celltype"] for ms in meta_s["cells"]]

    mmap_fname = opj(scsrh_repl_root,  f"{slide_id}_cells.dat")

    fd = np.memmap(mmap_fname,
                    dtype="uint16",
                    mode="r",
                    shape=tuple(meta_s["tensor_shape"]))
    im = np.array(fd, dtype=float)
    fd._mmap.close()
    del fd

    im = torch.from_numpy(im).to(torch.float32).contiguous()

    nuclei_im = im[[c=="nuclei" for c in celltypes]]
    mp_im = im[[c=="macrophage" for c in celltypes]]
    
    if len(nuclei_im):
        nuclei_mean = (nuclei_im.mean(dim=0), nuclei_im.shape[0])
    else:
        nuclei_mean = (torch.zeros((3,64,64)), 0)
        
    if len(mp_im):
        mp_mean = (mp_im.mean(dim=0), mp_im.shape[0])
    else:
        mp_mean = (torch.zeros((3,64,64)), 0)
        
    return nuclei_mean, mp_mean


In [ ]:
im_means = [get_mean(r) for i, r in tqdm.tqdm(sampled_train_slides.iterrows(), total=len(sampled_train_slides))]

In [ ]:
torch.save(im_means, "all_im_means.pt")

In [ ]:
def get_im_center(im_means_tuple_tt):
    nu_means_tt = torch.stack([i[0][0] for i in im_means_tuple_tt])
    nu_wts_tt = torch.tensor([i[0][1] for i in im_means_tuple_tt]).to(torch.float)
    
    mp_means_tt = torch.stack([i[1][0] for i in im_means_tuple_tt])
    mp_wts_tt = torch.tensor([i[1][1] for i in im_means_tuple_tt]).to(torch.float)
    
    nu_means_tt = einops.einsum(nu_wts_tt, nu_means_tt, "ns, ns d h w -> d h w") / sum(nu_wts_tt)
    mp_means_tt = einops.einsum(mp_wts_tt, mp_means_tt, "ns, ns d h w -> d h w") / sum(mp_wts_tt)
    
    return nu_means_tt, mp_means_tt

all_class_mean = {}
all_tumor_types = set(sampled_train_slides["label"])
fig, axs = plt.subplots(2, len(all_tumor_types) + 1, figsize=(10,2))

for i, tumortype in enumerate(all_tumor_types):
    tumortype_idx = torch.tensor(sampled_train_slides["label"] == tumortype).tolist()
    im_means_tuple_tt = [i for i,c in zip(im_means,tumortype_idx) if c]
    nu_means_tt, mp_means_tt = get_im_center(im_means_tuple_tt)

    all_class_mean[tumortype] = {
        "nu":nu_means_tt,
        "mp":mp_means_tt
    }
    
    viz_nu = rearrange_viz(normalize_im(aug(nu_means_tt[:2,...])))
    viz_mp = rearrange_viz(normalize_im(aug(mp_means_tt[:2,...])))
    
    #Image.fromarray(viz_nu.numpy()).save(f"nu_{tumortype}.png")
    #Image.fromarray(viz_mp.numpy()).save(f"mp_{tumortype}.png")
    
    axs[0][i].imshow(viz_nu)
    axs[1][i].imshow(viz_mp) 

nu_means_all, mp_means_all = get_im_center(im_means)
    
all_class_mean["all"] = {
        "nu":nu_means_all,
        "mp":mp_means_all
    }

viz_nu = rearrange_viz(normalize_im(aug(nu_means_all[:2,...])))
viz_mp = rearrange_viz(normalize_im(aug(mp_means_all[:2,...])))

#Image.fromarray(viz_nu.numpy()).save(f"nu_all.png")
#Image.fromarray(viz_mp.numpy()).save(f"mp_all.png")

axs[0][-1].imshow(viz_nu)
axs[1][-1].imshow(viz_mp) 

torch.save(all_class_mean, "all_tt_means.pt")

In [ ]:
im_means = torch.load("all_tt_means.pt")
